# Конспект. Модуль 1: Деревья решений для ML (CART)

## 1. Зачем это нужно и что мы на самом деле строим

Вы уже знаете дерево как структуру данных (Неделя 4): узлы, рёбра, обходы DFS/BFS, BST, где значения хранятся в узлах и порядок задаётся вручную логикой вставки. **Дерево решений в ML — это другая сущность**, хотя и растёт по похожему принципу «узел -> два потомка».

Разница принципиальная:
- В BST правило разбиения задано заранее человеком: «если ключ меньше — влево».
- В дереве решений **правило разбиения ищет сам алгоритм**, перебирая варианты и выбирая тот, который лучше всего разделяет данные по целевой переменной.

Аналогия: представьте игру «20 вопросов», где вы отгадываете объект, задавая да/нет-вопросы. Хороший вопрос сразу отсекает половину вариантов («это живое?»), плохой — почти ничего не отсекает («это весит меньше 900 кг?», если 99% объектов весят меньше). CART — это формализация того, как автоматически находить именно «хорошие вопросы» на основе данных.

**CART** (Classification And Regression Trees) — алгоритм, который:
1. Строит **бинарное** дерево (у каждого внутреннего узла ровно 2 потомка — в отличие от, например, ID3/C4.5, которые могут делать многосторонние разбиения).
2. На каждом шаге ищет пару **(признак, порог)**, которая лучше всего разделяет объекты в узле.
3. Рекурсивно повторяет это для каждого потомка, пока не выполнится критерий остановки.

Каждый **путь от корня до листа** — это конъюнкция условий («сумма > 5000 AND час < 3 AND …»), а **лист** хранит финальное предсказание: класс (или вероятность класса) для классификации, число — для регрессии.

## 2. Что значит «качество разбиения» — интуиция

Представим узел с 10 транзакциями, из которых 5 мошеннические и 5 честные — это **максимально «грязный» (impure)** узел: угадать класс наугад — то же самое, что подбросить монетку.

Теперь представим узел, где все 10 транзакций честные — это **чистый (pure)** узел: если объект попал сюда, мы можем предсказать «не мошенничество» с полной уверенностью.

**Задача дерева на каждом шаге — найти такое разбиение, после которого дети окажутся более «чистыми», чем родитель.** Осталось формализовать, что такое «чистота», числом. Для этого существует два основных критерия для классификации: **Gini impurity** и **Entropy**.

## 3. Критерии для классификации

### 3.1. Gini impurity

Для узла *t*, где `p_k` — доля объектов класса *k* среди объектов, попавших в этот узел (K классов всего):

In [ ]:
Gini(t) = 1 - Σ(k=1..K) p_k²

**Как читать формулу:** если бы мы наугад выбрали объект из узла и наугад присвоили ему класс согласно распределению `p_k`, Gini — это вероятность **ошибиться** при таком случайном угадывании.

**Граничные случаи (бинарная классификация, K=2):**
- Узел чистый: `p_0=1, p_1=0` -> `Gini = 1 - (1² + 0²) = 1 - 1 = 0` (минимум, полная чистота).
- Узел максимально грязный: `p_0=0.5, p_1=0.5` -> `Gini = 1 - (0.25 + 0.25) = 0.5` (максимум для 2 классов).

**Численный пример.** Узел с 5 честными и 5 мошенническими транзакциями: `p_честн=0.5, p_мошен=0.5` -> `Gini = 1 - (0.5² + 0.5²) = 1 - 0.5 = 0.5`.

Узел с 9 честными и 1 мошеннической: `p_честн=0.9, p_мошен=0.1` -> `Gini = 1 - (0.81 + 0.01) = 1 - 0.82 = 0.18`. Заметно чище, чем 0.5.

### 3.2. Entropy и Information Gain

In [ ]:
Entropy(t) = - Σ(k=1..K) p_k · log2(p_k)

**Интуиция логарифма:** это мера «неопределённости» в теоретико-информационном смысле — сколько бит в среднем нужно, чтобы «закодировать» класс случайно выбранного объекта из узла. Чем ближе распределение классов к равномерному, тем больше бит нужно (больше неопределённости), чем более узел чист — тем меньше.

**Граничные случаи (K=2):**
- Чистый узел: `p_0=1` -> `log2(1)=0`, а `p_1=0` по соглашению даёт вклад `0 · log2(0) = 0` -> `Entropy = 0`.
- 50/50: `Entropy = -(0.5·log2(0.5) + 0.5·log2(0.5)) = -(0.5·(-1) + 0.5·(-1)) = 1` (максимум для 2 классов, в отличие от Gini, максимум entropy равен 1, а не 0.5 — это просто разный масштаб, качественно кривые похожи).

**Information Gain** — на сколько энтропия уменьшилась после разбиения родителя на детей:

In [ ]:
IG = Entropy(parent) - [ (N_left/N) · Entropy(left) + (N_right/N) · Entropy(right) ]

где `N` — число объектов в родителе, `N_left`/`N_right` — в левом/правом потомке. Дерево выбирает разбиение с **максимальным** IG (эквивалентно — с максимальным падением impurity, если считать через Gini вместо Entropy — тогда говорят просто про «Gini Gain», механика та же).

### 3.3. Gini vs Entropy — что выбрать на практике

- Кривые Gini и Entropy как функции от `p` качественно похожи (обе — «горка» с максимумом на 0.5 и нулём на краях), поэтому на практике они **почти всегда выбирают одинаковые разбиения**.
- Entropy чуть тяжелее вычислительно (логарифм против умножения), поэтому Gini — критерий по умолчанию в sklearn (`criterion='gini'`).
- Различия становятся заметны в основном в узкой полосе распределений и почти никогда не влияют на итоговое качество модели заметно — не стоит тратить много времени на выбор между ними, это не значимый гиперпараметр для тюнинга.

## 4. Регрессионные деревья: MSE и MAE

Для регрессии нет «классов», значит нет `p_k` — критерий меняется на **разброс значений таргета внутри узла**.

**MSE-критерий (squared_error):**

In [ ]:
MSE(t) = (1/N_t) · Σ(i∈t) (y_i - ȳ_t)²

где `ȳ_t` — среднее значение таргета по всем объектам узла *t*. Это просто **дисперсия** таргета внутри узла. Дерево выбирает разбиение, минимизирующее взвешенную сумму MSE детей (что эквивалентно максимизации «уменьшения дисперсии» — полный аналог Information Gain, только для непрерывного таргета).

**Важно:** предсказание регрессионного дерева в листе — это **среднее** значение таргета среди обучающих объектов, попавших в этот лист. Дерево физически не может предсказать значение, которого не было близко к средним обучающей выборки — это ступенчатая, кусочно-постоянная функция.

**MAE-критерий (absolute_error):** использует медиану вместо среднего и модуль отклонения вместо квадрата:

In [ ]:
MAE(t) = (1/N_t) · Σ(i∈t) |y_i - median_t|

Более устойчив к выбросам (один экстремальный `TransactionAmt` не утянет разбиение так сильно, как при MSE), но заметно медленнее на больших данных (медиану сложнее пересчитывать инкрементально при переборе порогов, чем среднее).

## 5. Алгоритм CART пошагово

Вот что происходит **внутри** `.fit()`, когда вы вызываете `DecisionTreeClassifier().fit(X, y)`:

1. **Начало:** все объекты — в одном узле (корне).
2. **Для текущего узла**, для **каждого признака** `j`:
   - Отсортировать значения признака `j` по возрастанию.
   - Сформировать кандидатов на порог — как правило, середины между соседними уникальными отсортированными значениями (например, если отсортированные значения `[10, 20, 30]`, кандидаты порогов — `15` и `25`).
   - Для каждого кандидата порога посчитать impurity (Gini/Entropy/MSE) обоих потомков и итоговый прирост качества.
3. **Выбрать глобально лучшую пару (признак, порог)** — ту, что даёт максимальное падение impurity среди **всех** признаков и **всех** порогов.
4. **Разбить узел** на два потомка согласно найденному правилу.
5. **Рекурсивно повторить шаги 2–4** для каждого потомка отдельно.
6. **Остановиться**, когда сработает один из критериев остановки (раздел 6).

**Категориальные признаки:** классический CART умеет искать оптимальное разбиение категорий на два подмножества (например, `{Москва, СПб}` против `{остальные города}`), перебирая возможные группировки. Библиотека sklearn **этого не делает** — она ожидает уже закодированные числа (отсюда необходимость OneHot/Ordinal encoding перед `DecisionTreeClassifier`). Забегая вперёд: именно отсутствие нормальной встроенной поддержки категорий у классического CART/sklearn — одна из причин, почему появился CatBoost (Модуль 8), который умеет работать с категориями нативно и умнее.

**Важное концептуальное замечание — жадность алгоритма.** CART на каждом шаге выбирает **локально лучшее** разбиение, не заглядывая вперёд. Построение **глобально оптимального** дерева решений — NP-полная задача, поэтому все практические алгоритмы (включая CART) — жадные эвристики. Это значит, что дерево иногда делает разбиение, которое выглядит слабым сейчас, но могло бы стать сильным в комбинации со следующим шагом — CART такую возможность **не видит**. Это не баг, а фундаментальное ограничение подхода — и одна из причин, почему **ансамбли** деревьев (Модуль 2 и далее) работают значительно лучше одного дерева.

**Сложность:** сортировка каждого признака в каждом узле стоит `O(N log N)`, всего признаков `M` -> на уровень дерева уходит примерно `O(M · N log N)`. Для больших датасетов (миллионы строк, сотни признаков) это дорого — именно поэтому современные библиотеки (Модуль 7, LightGBM) заменяют точный перебор на **гистограммный** — бинируют значения признака заранее и перебирают не все уникальные значения, а десятки корзин. Держите этот момент в голове — он станет ключевым в Модуле 7.

## 6. Критерии остановки (без них дерево росло бы до идеального переобучения)

Дерево остановит рост узла, если выполняется любое из:
- Узел стал **чистым** (impurity = 0, все объекты одного класса / одно значение).
- Достигнута **максимальная глубина** (`max_depth`).
- В узле осталось **меньше объектов**, чем `min_samples_split` — разбивать уже нельзя.
- Любое возможное разбиение дало бы потомка с числом объектов **меньше** `min_samples_leaf`.
- Максимальное падение impurity среди всех кандидатов **меньше** `min_impurity_decrease` — разбиение технически возможно, но не даёт значимого выигрыша.

Без этих ограничений (`max_depth=None` и остальное по умолчанию) дерево будет расти, пока каждый лист не станет чистым — то есть до полного запоминания обучающей выборки (в пределе — один лист на объект, если нет повторяющихся точек с разными классами).

## 7. Полный numeric-пример вручную

Возьмём игрушечный датасет из 8 транзакций с одним признаком `hour` (час совершения транзакции) и таргетом `is_fraud`:

| # | hour | is_fraud |
|---|------|----------|
| 1 | 1  | 1 |
| 2 | 2  | 1 |
| 3 | 3  | 1 |
| 4 | 9  | 0 |
| 5 | 10 | 0 |
| 6 | 11 | 0 |
| 7 | 14 | 0 |
| 8 | 23 | 1 |

**Шаг 1. Gini корня.** Всего 8 объектов: 4 мошеннических (`hour` = 1,2,3,23), 4 честных. `p_1=0.5, p_0=0.5` -> `Gini(root) = 1 - (0.5² + 0.5²) = 0.5`.

**Шаг 2. Кандидаты порогов.** Отсортированные значения `hour`: 1, 2, 3, 9, 10, 11, 14, 23. Середины соседних пар: 1.5, 2.5, 6, 9.5, 10.5, 12.5, 18.5 — итого 7 кандидатов.

**Шаг 3. Считаем Gini для нескольких кандидатов** (порог означает `hour ≤ t -> влево`):

- `t = 1.5`: слева {1} -> 1 объект, класс 1 -> `Gini_left=0`. Справа {2,3,9,10,11,14,23} -> 7 объектов, из них 2 мошеннических (2,3,23... стоп, hour=2 и 3 попали влево? нет, `hour ≤ 1.5` включает только hour=1). Справа: 2,3,9,10,11,14,23 -> мошеннических среди них: 2,3,23 -> 3 из 7. `p=3/7≈0.429` -> `Gini_right = 1-(0.429²+0.571²) ≈ 1-(0.184+0.327)=0.489`. Взвешенный Gini = `(1/8)·0 + (7/8)·0.489 ≈ 0.428`.

- `t = 6` (разделяет {1,2,3} и {9,10,11,14,23}): слева 3 объекта, все мошеннические -> `Gini_left=0`. Справа 5 объектов, 1 мошеннический (hour=23) -> `p=1/5=0.2` -> `Gini_right=1-(0.04+0.64)=0.32`. Взвешенный Gini = `(3/8)·0 + (5/8)·0.32 = 0.2`.

- `t = 18.5` (разделяет {1,2,3,9,10,11,14} и {23}): слева 7 объектов, 3 мошеннических -> `p=3/7≈0.429` -> `Gini_left≈0.489`. Справа 1 объект, класс 1 -> `Gini_right=0`. Взвешенный Gini = `(7/8)·0.489 + (1/8)·0 ≈ 0.428`.

**Сравниваем:** `t=6` даёт взвешенный Gini `0.2` — заметно лучше, чем `t=1.5` (0.428) и `t=18.5` (0.428). Если проверить остальные кандидаты (9.5, 10.5, 12.5), они окажутся ещё хуже, так как будут дробить однородную группу {9,10,11,14,0} без пользы. **Итог: лучшее разбиение корня — `hour ≤ 6`.**

**Дерево после первого разбиения:**

In [ ]:
                 [hour ≤ 6?]
                 /          \
             да /            \ нет
               /              \
      {1,2,3}, все fraud=1    {9,10,11,14,23}, 4 честных + 1 fraud (hour=23)
      Gini = 0 -> ЛИСТ,         Gini = 0.32 -> узел неоднородный,
      predict fraud=1          продолжаем разбивать

**Шаг 4. Разбиваем правый узел** {9,10,11,14,23} (5 объектов, 1 мошеннический). Кандидаты порогов: 9.5, 10.5, 12.5, 18.5. Проверим `t=18.5`: слева {9,10,11,14} — все честные -> `Gini_left=0`. Справа {23} — мошенник -> `Gini_right=0`. Взвешенный Gini = `(4/5)·0 + (1/5)·0 = 0`. **Идеальное разбиение** — оба потомка чистые.

**Финальное дерево:**

In [ ]:
                     [hour ≤ 6?]
                    /            \
                  да              нет
                  /                \
        ЛИСТ: fraud=1        [hour ≤ 18.5?]
        (3 объекта)          /              \
                            да                нет
                            /                  \
                   ЛИСТ: fraud=0        ЛИСТ: fraud=1
                   (4 объекта)          (1 объект)

Это, конечно, искусственно чистый пример (в реальных данных идеальных разбиений почти не бывает) — но именно так, механически, шаг за шагом, CART строит дерево на любых данных: перебор кандидатов -> выбор лучшего -> рекурсия.

## 8. Гиперпараметры: что они физически ограничивают

| Параметр | Что делает | Эффект при увеличении | Эффект при уменьшении |
|---|---|---|---|
| `max_depth` | Максимальная глубина дерева | Больше глубина -> сложнее модель, риск переобучения | Меньше глубина -> проще модель, риск недообучения |
| `min_samples_split` | Мин. объектов в узле, чтобы его вообще пробовать разбивать | Больше -> дерево грубее, меньше листьев | Меньше -> дерево детальнее, ближе к переобучению |
| `min_samples_leaf` | Мин. объектов в **каждом** итоговом листе | Больше -> сглаживает предсказания, защита от переобучения на выбросах | Меньше (вплоть до 1) -> лист может состоять из одного объекта — сильное переобучение |
| `max_features` | Сколько признаков рассматривать при поиске лучшего разбиения в узле | Все признаки -> детерминированный полный перебор | Меньше признаков -> элемент случайности (пригодится в Random Forest, Модуль 2) |
| `criterion` | Формула impurity: `gini`/`entropy`/`log_loss` (классификация), `squared_error`/`absolute_error` (регрессия) | — | — |
| `min_impurity_decrease` | Минимальный порог полезности разбиения | Больше -> дерево не делает «слабые» разбиения, проще | Меньше (0 по умолчанию) -> разбивает при любом, даже минимальном выигрыше |
| `ccp_alpha` | Коэффициент штрафа за сложность при пост-стрижке (раздел 9) | Больше -> дерево после обучения агрессивно подрезается | 0 (по умолчанию) -> стрижки нет |

**Практическое правило:** `min_samples_leaf` и `max_depth` — два самых влияющих на переобучение параметра для одиночного дерева, с них и стоит начинать тюнинг, если бы вы тюнили одиночное дерево (на практике для ансамблей это будет `num_leaves`/`min_data_in_leaf` в LightGBM — прямые аналоги, см. Модуль 7).

## 9. Bias-Variance для одного дерева — почему это «нестабильный» алгоритм

Вспомните Bias-Variance Tradeoff (уже знакомая тема, Неделя 4) — применим её конкретно к деревьям:

**Глубокое дерево без ограничений (`max_depth=None`, `min_samples_leaf=1`):**
- **Bias почти нулевой** — дерево способно идеально подстроиться под обучающую выборку вплоть до каждой точки (если нет двух объектов с одинаковыми признаками, но разными классами — тогда каждый лист чист).
- **Variance огромный** — дерево **крайне чувствительно** к небольшим изменениям в данных. Уберите/добавьте пару строк — и уже на первом же уровне может выбраться **другой** признак и порог для разбиения (особенно если несколько кандидатов давали близкий Gini), а значит, вся структура дерева ниже этого узла изменится кардинально. Эта особенность называется **нестабильностью** (instability) — деревья считаются одним из самых нестабильных базовых алгоритмов в ML.

**Неглубокое дерево (`max_depth=2-3`):**
- **Bias высокий** — простых правил часто не хватает, чтобы уловить сложную зависимость.
- **Variance низкий** — структура дерева меняется несильно при небольших изменениях данных.

**Почему это важно за пределами Модуля 1:** именно нестабильность одиночного дерева — это то свойство, которое **эксплуатируют** все ансамблевые методы, но по-разному:
- **Bagging/Random Forest (Модуль 2)** усредняет много нестабильных высоко-дисперсных деревьев — усреднение гасит variance, оставляя bias примерно на том же уровне, что и у одного глубокого дерева.
- **Бустинг (Модули 3–8)**, наоборот, специально использует **неглубокие, высоко-смещённые** деревья (`max_depth=3-8` типично) и последовательно снижает bias, добавляя новые деревья, которые исправляют остаточную ошибку.

Это ключевая связка, которую стоит держать в голове весь курс: **одно и то же базовое дерево используется в bagging и в boosting принципиально по-разному именно из-за этого bias-variance разложения.**

## 10. Pruning (стрижка дерева)

**Pre-pruning (предварительная остановка)** — это всё, что мы уже разобрали в разделе 8: ограничить рост дерева гиперпараметрами **до и во время** обучения (`max_depth`, `min_samples_leaf` и т.д.). Просто, дёшево, но может остановить рост слишком рано, пропустив полезное разбиение, которое стало бы видно только на следующем уровне.

**Post-pruning (стрижка после обучения)** — вырастить дерево **полностью** (без ограничений), а затем обрезать лишние ветви. Классический метод — **Cost-Complexity Pruning (CCP)**:

In [ ]:
R_alpha(T) = R(T) + alpha · |leaves(T)|

где `R(T)` — суммарная ошибка дерева на обучающей выборке, `|leaves(T)|` — число листьев, `alpha` (в sklearn — `ccp_alpha`) — коэффициент штрафа за каждый дополнительный лист. При `alpha=0` штрафа нет — оптимальным будет полное, максимально переобученное дерево. При увеличении `alpha` становится выгодно жертвовать точностью на train ради меньшего числа листьев — алгоритм итеративно «схлопывает» те поддеревья, у которых прирост качества на объект штрафа не окупает.

Практически: `sklearn.tree.DecisionTreeClassifier` имеет метод `cost_complexity_pruning_path`, который возвращает набор кандидатов `alpha`, а дальше — обычный подбор по валидации (какой `alpha` даёт лучшее качество на holdout).

**Важная оговорка на будущее:** pruning критичен, если вы **действительно** используете одиночное дерево как финальную модель. Внутри ансамблей (бустинг, бэггинг) отдельные деревья почти всегда используются **неглубокими и без сложной стрижки** — переобучение всего ансамбля контролируется другими механизмами (learning rate, число деревьев, регуляризация на уровне ансамбля — Модули 5, 9). Не удивляйтесь, что в LightGBM/CatBoost вы не увидите явного `ccp_alpha` — там работают другие рычаги.

## 11. Практика: код

### 11.1. Классификация — переобучение в лицах

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
import matplotlib.pyplot as plt

X, y = make_classification(n_samples=1000, n_features=10,
                            n_informative=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

depths = range(1, 20)
train_acc, test_acc = [], []

for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    train_acc.append(clf.score(X_train, y_train))
    test_acc.append(clf.score(X_test, y_test))

plt.plot(depths, train_acc, label="train")
plt.plot(depths, test_acc, label="test")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.legend()
plt.title("Классический переобучающий график: train растёт, test — плато и падение")
plt.show()

**Что искать на графике:** train-accuracy будет монотонно расти к 1.0 (дерево запоминает выборку), test-accuracy вырастет до какого-то `max_depth`, а затем начнёт падать или колебаться — это и есть точка, где дерево из «недообученного» переходит в «переобученное». Именно этот разрыв между кривыми — визуальное определение переобучения, которое пригодится в каждом следующем модуле курса.

### 11.2. Визуализация дерева

In [ ]:
from sklearn.tree import plot_tree

clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train, y_train)

plt.figure(figsize=(16, 8))
plot_tree(clf, filled=True, feature_names=[f"f{i}" for i in range(10)],
          class_names=["0", "1"], fontsize=8)
plt.show()

В каждом узле визуализации sklearn покажет: правило разбиения, значение `gini`, число объектов (`samples`), распределение по классам (`value`) — можно вручную сверить с формулами из разделов 3 и 7.

### 11.3. Регрессия

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.datasets import make_regression

X, y = make_regression(n_samples=500, n_features=5, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

reg = DecisionTreeRegressor(max_depth=4, min_samples_leaf=10, random_state=42)
reg.fit(X_train, y_train)

print("Train MSE:", ((reg.predict(X_train) - y_train) ** 2).mean())
print("Test MSE:", ((reg.predict(X_test) - y_test) ** 2).mean())

Обратите внимание на `min_samples_leaf=10` — без этого регрессионное дерево на шумных данных (`noise=10`) склонно создавать листья из 1-2 объектов, идеально подгоняясь под случайный шум конкретных точек.

### 11.4. Забег вперёд: feature_importances_

In [ ]:
import numpy as np
importances = clf.feature_importances_
for name, imp in sorted(zip([f"f{i}" for i in range(10)], importances),
                         key=lambda x: -x[1]):
    print(f"{name}: {imp:.3f}")

Это версия **Gain**-важности (суммарное падение impurity по всем разбиениям, где участвовал признак, взвешенное на долю объектов) — сейчас достаточно знать, что она существует; подробный разбор разных видов важности признаков будет в Модуле 11.

## 12. Частые вопросы на собеседовании по теме

| Вопрос | На что обратить внимание в ответе |
|---|---|
| Почему дерево решений называют «жадным» алгоритмом? | Локально оптимальный выбор на каждом шаге, без заглядывания вперёд; глобально оптимальное дерево — NP-полная задача |
| Чем Gini отличается от Entropy на практике? | Похожая форма кривой, Entropy чуть дороже вычислительно (log), на практике почти не влияет на выбор разбиений |
| Почему одиночное дерево — modель с высокой дисперсией? | Малые изменения данных -> другое разбиение на верхних уровнях -> полностью другая структура ниже |
| Как дерево решений обрабатывает пропуски (NaN)? | Зависит от реализации: sklearn исторически не поддерживал NaN напрямую (нужна импутация), CART в оригинале и некоторые современные библиотеки используют «суррогатные разбиения» |
| Что будет, если признак константный (не варьируется) в узле? | Он никогда не будет выбран для разбиения — падение impurity по нему всегда равно нулю |
| Как дерево строит регрессию, если это «ступенчатая» функция? | Каждый лист предсказывает среднее (или медиану) — предсказание кусочно-постоянно, не может экстраполировать за пределы диапазона обучающих значений |

## 13. Чек-поинт — попробуйте ответить без подсказок

1. Почему Gini и Entropy почти всегда дают похожий, но не идентичный результат?
2. Что произойдёт с деревом без `max_depth` и `min_samples_leaf` на шумных данных?
3. Почему построение **глобально оптимального** дерева решений — вычислительно неразрешимая задача на практике, и как CART обходит эту проблему?
4. Почему регрессионное дерево не может предсказать значение, большее максимума обучающей выборки, даже если новый объект имеет экстремальные значения признаков?
5. Объясните своими словами, почему одиночное дерево называют «нестабильным» алгоритмом и почему это свойство критично важно для понимания Bagging (следующий модуль).

## Ответы для самопроверки

<details>
<summary>Раскрыть после того, как попробуете ответить сами</summary>

1. Обе формулы измеряют «неоднородность» распределения классов и имеют качественно одинаковую форму (ноль на чистых узлах, максимум на равномерном распределении), но Entropy использует логарифм (чувствительнее к малым `p`), а Gini — квадрат (чуть более «сглаженная» функция) — на практике разница в выбираемых разбиениях минимальна.
2. Дерево будет расти до полной чистоты каждого листа, подстраиваясь под шум обучающей выборки вплоть до отдельных выбросов — классическое сильное переобучение: train-метрика близка к идеальной, test-метрика заметно хуже.
3. Число возможных бинарных деревьев над данным набором признаков и порогов растёт комбинаторно с числом объектов и признаков — полный перебор всех структур невозможен даже для скромных датасетов. CART обходит проблему, выбирая на каждом шаге **локально** лучшее разбиение (жадная эвристика) вместо перебора всех возможных деревьев целиком.
4. Потому что лист хранит среднее (или медиану) обучающих таргетов, попавших в него — это число не может выйти за диапазон обучающих значений; дерево физически не умеет экстраполировать, только интерполировать внутри уже виденных диапазонов.
5. Нестабильность означает, что небольшое изменение обучающей выборки может привести к выбору другого признака/порога на верхних уровнях, из-за чего вся нижележащая структура меняется кардинально — это высокая дисперсия отдельной модели. Bagging строит много таких нестабильных деревьев на разных bootstrap-выборках и **усредняет** их предсказания — усреднение независимых высоко-дисперсных оценок статистически снижает итоговую дисперсию, почти не трогая bias. Если бы дерево было стабильным (низкая дисперсия), усреднение копий друг друга не дало бы почти никакого выигрыша — именно поэтому Random Forest эффективен именно на деревьях, а не, скажем, на линейной регрессии.

</details>